In [ ]:
!pip install hazm
!pip install tensorflow_gpu==2.0

     |████████████████████████████████| 316 kB 12.7 MB/s 
     |████████████████████████████████| 1.4 MB 27.1 MB/s 
     |████████████████████████████████| 233 kB 39.5 MB/s 
  Created wheel for nltk: filename=nltk-3.3-py3-none-any.whl size=1394488 sha256=86db3bb98b8c966a5521fb1111ced856dab731c7ff3e73dcb5ed2440e6722663
  Stored in directory: /root/.cache/pip/wheels/9b/fd/0c/d92302c876e5de87ebd7fc0979d82edb93e2d8d768bf71fac4
  Created wheel for libwapiti: filename=libwapiti-0.2.1-cp37-cp37m-linux_x86_64.whl size=154538 sha256=0b9c3334345afdc01043b45b576943a44b6cb3465aff56e271d5dc94c41bf621
  Stored in directory: /root/.cache/pip/wheels/ab/b2/5b/0fe4b8f5c0e65341e8ea7bb3f4a6ebabfe8b1ac31322392dbf
Successfully built nltk libwapiti
  Attempting uninstall: nltk
    Found existing installation: nltk 3.2.5
    Uninstalling nltk-3.2.5:
      Successfully uninstalled nltk-3.2.5
     |████████████████████████████████| 380.8 MB 40 kB/s 
     |████████████████████████████████| 3.8 MB 29.6 MB/s 
    

In [ ]:
import numpy as np
import pandas as pd
from tensorflow import keras
from tensorflow.keras import optimizers
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Dense, Input, Embedding, Dropout, GlobalMaxPool1D, Conv1D, MaxPooling1D, Conv2D, MaxPooling2D, SpatialDropout1D
from tensorflow.keras.layers import LSTM, Bidirectional, GRU, BatchNormalization
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from hazm import *
import re

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#dataset=pd.read_csv('/content/drive/MyDrive/textmining/digimg/train.csv')
dataset=pd.read_excel('/content/drive/MyDrive/dataset1/train-dev.xlsx')
#dataset=pd.read_csv('/content/drive/MyDrive/textmining/train3.csv', encoding="utf-8")
#dataset=pd.read_csv('/content/drive/MyDrive/textmining/train3.csv', encoding="utf-8")
#dataset=pd.read_table("/content/drive/MyDrive/textmining/train4.txt", delimiter=r"--")
dataset

,number,content,label,label_id
0,0,به گزارش خبرنگار حوزه بهداشت و درمان گروه علمی...,پزشکی,7
1,1,به گزارش خبرنگار فوتبال و فوتسال گروه ورزشی با...,ورزشی,6
2,2,بهروز اکرمی، در گفتگو با خبرنگار اجتماعی باشگا...,اجتماعی,0
3,3,به گزارش خبرنگار حوزه شهری گروه اجتماعی باشگاه...,اجتماعی,0
4,4,به گزارش باشگاه خبرنگاران و به نقل از روابط عم...,فرهنگی هنری,5
...,...,...,...,...
14770,1475,به گزارش خبرنگارکلینیک باشگاه خبرنگاران؛ کم خو...,پزشکی,7
14771,1476,رضا قدیمی رئیس کمیته منابع انسانی ستاد اربعین ...,اجتماعی,0
14772,1477,به گزارش خبرنگار گروه علمی و دانشگاهی خبرگزاری...,علمی فناوری,4
14773,1478,به گزارش خبرنگار حوزه کلینیک گروه علمی پزشکی ب...,پزشکی,7


In [ ]:
sentence_train=dataset['content']
label_train=dataset['label_id']
print("Numbers of training sentences", sentence_train.shape)
print("Numbers of training label", label_train.shape)

Numbers of training sentences (14775,)
Numbers of training label (14775,)


In [ ]:
from collections import Counter
cnt= Counter(label_train)
cnt= dict(cnt)
print(cnt)

{7: 1876, 6: 1243, 0: 1947, 5: 2301, 4: 2192, 3: 2035, 2: 1774, 1: 1407}


In [ ]:
#convert dataframes to numpy arrays
sentence_train=np.asarray(sentence_train)
label_train=np.asarray(label_train)

In [ ]:
label_train

array([7, 6, 0, ..., 4, 7, 1])

In [ ]:
#datasett=pd.read_csv('/content/drive/MyDrive/textmining/digimg/persian.txt',delimiter='\t')
datasett=pd.read_csv('/content/drive/MyDrive/dataset1/persian.txt',delimiter='\t')
stops=list(datasett['value'])
stop_words = set((stops))

In [ ]:
!pip install -q clean-text[gpl]
from cleantext import clean 

     |████████████████████████████████| 64 kB 2.2 MB/s 
     |████████████████████████████████| 184 kB 24.1 MB/s 
     |████████████████████████████████| 241 kB 40.3 MB/s 


In [ ]:
#preprocessing


normalizer=Normalizer()
lemmatizer=Lemmatizer()
stemmer=Stemmer()
#turn a doc into clean token
def clean_data(doc):
  doc=normalizer.normalize(doc)
  tokenized=word_tokenize(doc)
  #print(tokenized)
  tokens=[]
  for token in tokenized:
    token=re.sub('[a-zA-Z0-9]+',"",token)
    token=re.sub("[.:,()/+]","",token)
    token=re.sub(r"\!+","!",token)
    token=re.sub(r"\؟+","؟",token)
    token = token.replace("#",'')
    token = token.replace("\u200c",'')
    token=re.sub("\u200c","",token)
    token=re.sub("[۰۱۲۳۴۵۶۷۸۹]+"," ",token)
    token = token.replace('[۰۱۲۳۴۵۶۷۸۹]+','')
    token = token.replace('[a-zA-Z0-9]+','')
   
    
    
    
    # removing wierd patterns
    wierd_pattern = re.compile("["
        u"\U0001F600-\U0001F64F"  # emoticons
        u"\U0001F300-\U0001F5FF"  # symbols & pictographs
        u"\U0001F680-\U0001F6FF"  # transport & map symbols
        u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
        u"\U00002702-\U000027B0"
        u"\U000024C2-\U0001F251"
        u"\U0001f926-\U0001f937"
        u'\U00010000-\U0010ffff'
        u"\u200d"
        u"\u2640-\u2642"
        u"\u2600-\u2B55"
        u"\u23cf"
        u"\u23e9"
        u"\u231a"
        u"\u3030"
        u"\ufe0f"
        u"\u2069"
        u"\u2066"
        # u"\u200c"
        u"\u2068"
        u"\u2067"
        "]+", flags=re.UNICODE)
    
    #token = wierd_pattern.sub(r'', token)
    cleanr = re.compile('<.*?>')
    token = re.sub(cleanr, '', token)
    # removing extra spaces, hashtags
    token = re.sub("[#+]", "", token)
    token = re.sub("$", "", token)
    token = re.sub("\s+", "", token)
     #if token.contain()
    
    tokens.append(token)
    #print(tokens)
  tokens=[w for w in tokens if not len(w)<=1]
  tokens=[w for w in tokens if not w.isdigit()]
  tokens=[lemmatizer.lemmatize(w) for w in tokens ]
  tokens=[w for w in tokens if w not in stop_words]
  #if token not in stop_words:
     # tokens.append(token)
  #tokens=[stemmer.stem(w) for w in tokens ]
  #print(tokens)
  tokens=' '.join(tokens)
  tokens = re.sub("[#]", "", tokens)
  tokens = re.sub("[$]", "", tokens)
  return tokens

In [ ]:
#apply preprocessing
train_docs=np.empty_like(sentence_train)
for index, document in enumerate(sentence_train):
  train_docs[index]=clean_data(document)

In [ ]:
train_docs[2]

'بهروز اکرم گفتگو خبرنگار اجتماع باشگاه خبرنگار گفتگو مجموعه وزارت تعاون کار رفاه اجتماع جمله ارائه تسهیلات واحد اقتصاد حمایت میکند راستا اداره تعاون کار رفاه اجتماع استان گسترش کارگاه واحد تولید طرح منظور افزایش تولیدات ایجاد اشتغال نیاز حمایت داشتدار برطرف شدن مشکلات تولیدکنندگان کارگر همت توان کار گرفتگیر ادامه اشاره ظرفیت فراوان تولید صنعتی استان هرمزگان اشاره کردکن تصریح کردکن حداکثر استفاده ظرفیت موجود صورت گرفتگیر زیرا اقدام مشکلات اشتغال رفع میشود مدیر تعاون کار رفاه اجتماع استان هرمزگان خاطرنشان کردکن سال اقتصاد فرهنگ عزم ملی مدیریت جهاد سعی کردکن مشکلات واحد صنعتی تولید توجه قرار گرفتگیر اداره راستا شعار ملی رفع مشکلات واحد صنعتی کارآفرین تولید کنندگان همراه است انتهای پیام'

In [ ]:
#set tokenizer and encode sentences
num_words=2000
#create the tokenizer
tokenizer=Tokenizer(num_words=num_words)
tokenizer.fit_on_texts(train_docs)
encoded_docs=tokenizer.texts_to_sequences(train_docs)

In [ ]:
#find maximum length of training sentences
max_length=max([len(s.split()) for s in train_docs])
max_length

4382

In [ ]:
vocab_size=len(tokenizer.word_index)
vocab_size

75807

In [ ]:
#pad embeded training sequences
x_train_padded=pad_sequences(encoded_docs, maxlen=max_length, padding='post')
x_train_padded[1]

array([ 14,   8, 250, ...,   0,   0,   0], dtype=int32)

## **GRU**

In [ ]:
model_blstm=Sequential()
model_blstm.add(Embedding(vocab_size,300, input_length=max_length))
model_blstm.add(GRU(300, return_sequences=True, name='GRU_layer'))
#model_blstm.add(SpatialDropout1D(0.2))
#model_blstm.add(BatchNormalization())
model_blstm.add(GlobalMaxPool1D())
model_blstm.add(Dropout(0.25))
model_blstm.add(Dense(300, activation="relu"))
model_blstm.add(Dropout(0.2))
model_blstm.add(Dense(8,activation='softmax'))

In [ ]:
#Optimizer = keras.optimizers.Adam(lr=0.001)
#model_blstm.compile(loss='categorical_crossentropy', optimizer=Optimizer, metrics=["categorical_accuracy"])
model_blstm.compile(loss='categorical_crossentropy', optimizer='adam', metrics=["categorical_accuracy"])
model_blstm.summary()
batch_size_blstm=64
epochs_blstm=4

Model: "sequential_3"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
embedding_3 (Embedding)      (None, 4382, 300)         22742100  
_________________________________________________________________
GRU_layer (GRU)              (None, 4382, 300)         541800    
_________________________________________________________________
global_max_pooling1d_3 (Glob (None, 300)               0         
_________________________________________________________________
dropout_6 (Dropout)          (None, 300)               0         
_________________________________________________________________
dense_6 (Dense)              (None, 300)               90300     
_________________________________________________________________
dropout_7 (Dropout)          (None, 300)               0         
_________________________________________________________________
dense_7 (Dense)              (None, 8)                

In [ ]:
categorical_label_train=to_categorical(label_train,8)
categorical_label_train

array([[0., 0., 0., ..., 0., 0., 1.],
       [0., 0., 0., ..., 0., 1., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 1.],
       [0., 1., 0., ..., 0., 0., 0.]], dtype=float32)

In [ ]:
hist_blstm=model_blstm.fit(x_train_padded, categorical_label_train,batch_size=batch_size_blstm, epochs=epochs_blstm, shuffle = True, validation_split = 0.1)

Train on 13297 samples, validate on 1478 samples
Epoch 1/4
13297/13297 [==============================] - 348s 26ms/sample - loss: 0.0676 - categorical_accuracy: 0.9788 - val_loss: 0.1056 - val_categorical_accuracy: 0.9594
Epoch 2/4
13297/13297 [==============================] - 348s 26ms/sample - loss: 0.0392 - categorical_accuracy: 0.9889 - val_loss: 0.1598 - val_categorical_accuracy: 0.9540
Epoch 3/4
13297/13297 [==============================] - 348s 26ms/sample - loss: 0.0275 - categorical_accuracy: 0.9906 - val_loss: 0.1289 - val_categorical_accuracy: 0.9601
Epoch 4/4
13297/13297 [==============================] - 347s 26ms/sample - loss: 0.0181 - categorical_accuracy: 0.9944 - val_loss: 0.1601 - val_categorical_accuracy: 0.9608


In [ ]:
test=pd.read_excel('/content/drive/MyDrive/dataset1/test.xlsx')
x_test=test['content']
y_test=test['label_id']
print("Number of testing sentence:", x_test.shape)
print("Number of testing label:", y_test.shape)

Number of testing sentence: (1642,)
Number of testing label: (1642,)


In [ ]:
x_test=np.asarray(x_test)
y_test=np.asarray(y_test)

In [ ]:
test_docs=np.empty_like(x_test)
for index, document in enumerate(x_test):
  test_docs[index]=clean_data(document)

In [ ]:
encoded_docs=tokenizer.texts_to_sequences(test_docs)
x_test_padded=pad_sequences(encoded_docs, maxlen=max_length, padding='post')
categorical_y_test=to_categorical(y_test,8)

In [ ]:
loss_blstm,acc_blstm=model_blstm.evaluate(x_test_padded, categorical_y_test, verbose=1)
print('Test Acurracy:%f'%(acc_blstm*100))

1642/1 [================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================

In [ ]:
y_pred = model_blstm.predict_classes(x_test_padded)

In [ ]:
content = []
true_label_id = []
pred_label_id = []
game = 0
shopping = 0
health = 0
IT = 0
Cultural = 0
Sports = 0
medical = 0
social=0
Economic=0
International=0
Political=0
for i in range(len(y_pred)):
  content.append(x_test[i])
  true_label_id.append(y_test[i])

  if y_pred[i] == 0:
    social +=1
    pred_label_id.append('social')
  elif y_pred[i] == 1:
    Economic +=1
    pred_label_id.append("Economic ")
  elif y_pred[i] == 2:
    International +=1
    pred_label_id.append("International")
  if y_pred[i] == 3:
    Political +=1
    pred_label_id.append("Political")
  elif y_pred[i] == 4:
    IT +=1
    pred_label_id.append("Scientific technology")
  elif y_pred[i] == 5:
    Cultural +=1
    pred_label_id.append("Cultural and artistic")
  elif y_pred[i] == 6:
    Sports +=1
    pred_label_id.append("Sports")
  elif y_pred[i] == 7:
    medical +=1
    pred_label_id.append("medical")

a=dataset['content'].count()
print("social: " , (social/a)*100)
print("Economic: " , (Economic/a)*100)
print("International: " , (International/a)*100)
print("Political: " , (Political/a)*100)
print("Scientific technology: " , (IT/a)*100)
print("Cultural and artistic: " , (Cultural/a)*100)
print("Sports: " , (Sports/a)*100)
print("medical: " , (medical/a)*100)

social:  1.4348561759729273
Economic:  1.0219966159052454
International:  1.3197969543147208
Political:  1.5431472081218274
Scientific technology:  1.6988155668358713
Cultural and artistic:  1.739424703891709
Sports:  0.9272419627749576
medical:  1.428087986463621
